# Login form with validation

This notebook builds a login form with `TextField` inputs and `@checks` validation rules. The sign-in button is disabled until the email and password pass their checks. Submitting sends a `login` action whose context carries the email.

Run the form cell, fill in the fields, then run the **wait** cell and click **Sign in**. The password is not included in the action context.

In [ ]:
from renderer_setup import new_surface

surface = new_surface("login-form")
surface.send(
    [
        {
            "version": "1.0",
            "updateDataModel": {
                "surfaceId": "login-form",
                "value": {"email": "", "password": "", "status": "Enter your details to sign in"},
            },
        },
        {
            "version": "1.0",
            "updateComponents": {
                "surfaceId": "login-form",
                "components": [
                    {"id": "root", "component": "Card", "child": "main_column"},
                    {
                        "id": "main_column",
                        "component": "Column",
                        "children": ["title", "email_field", "password_field", "login_btn", "status"],
                    },
                    {"id": "title", "component": "Text", "text": "## Welcome back"},
                    {
                        "id": "email_field",
                        "component": "TextField",
                        "label": "Email",
                        "value": {"@path": "/email"},
                        "@checks": [
                            {
                                "condition": {"@call": "required", "args": {"value": {"@path": "/email"}}},
                                "message": "Email is required",
                            },
                            {
                                "condition": {"@call": "email", "args": {"value": {"@path": "/email"}}},
                                "message": "Please enter a valid email address",
                            },
                        ],
                    },
                    {
                        "id": "password_field",
                        "component": "TextField",
                        "label": "Password",
                        "variant": "obscured",
                        "value": {"@path": "/password"},
                        "@checks": [
                            {
                                "condition": {"@call": "required", "args": {"value": {"@path": "/password"}}},
                                "message": "Password is required",
                            },
                            {
                                "condition": {
                                    "@call": "length",
                                    "args": {"value": {"@path": "/password"}, "min": 8},
                                },
                                "message": "Password must be at least 8 characters long",
                            },
                        ],
                    },
                    {
                        "id": "login_btn",
                        "component": "Button",
                        "child": "login_btn_text",
                        "variant": "primary",
                        "@checks": [
                            {
                                "condition": {
                                    "@call": "and",
                                    "args": {
                                        "values": [
                                            {"@call": "email", "args": {"value": {"@path": "/email"}}},
                                            {
                                                "@call": "length",
                                                "args": {"value": {"@path": "/password"}, "min": 8},
                                            },
                                        ]
                                    },
                                },
                                "message": "Please fix errors before signing in",
                            }
                        ],
                        "action": {
                            "event": {
                                "name": "login",
                                "context": {"email": {"@path": "/email"}},
                            }
                        },
                    },
                    {"id": "login_btn_text", "component": "Text", "text": "Sign in"},
                    {"id": "status", "component": "Text", "text": {"@path": "/status"}, "variant": "caption"},
                ],
            },
        },
    ]
)
surface

This cell waits for the **Sign in** click. Run it after the form is filled in.

In [ ]:
login_action = await surface.next_action()
login = login_action.model_dump(by_alias=True, exclude_unset=True)
login

Run this cell to show the result in the form. It replaces the status line with a message that uses the email from the action.

In [ ]:
email = login["action"]["context"]["email"]
surface.send(
    {
        "version": "1.0",
        "updateDataModel": {
            "surfaceId": "login-form",
            "path": "/status",
            "value": f"Signed in as {email}",
        },
    }
)